<a href="https://colab.research.google.com/github/awaiskhan005/Natural-Language-processing-NLP-CHATBOT-DEVELOPMENT/blob/main/AI_agent_for_Xact_financial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
import pandas as pd

# Sample Invoice and Bank Data (Real Data Simulation)
invoice_data = {
    'Invoice Number': ['INV001', 'INV002', 'INV003'],
    'Amount Due': [500, 300, 1000],
    'Due Date': ['2025-03-01', '2025-02-28', '2025-03-10'],
    'Client': ['Client A', 'Client B', 'Client C']
}

bank_data = {
    'Transaction ID': ['TX001', 'TX002', 'TX003'],
    'Amount Paid': [500, 300, 1000],
    'Payment Date': ['2025-02-28', '2025-02-28', '2025-03-01'],
    'Client': ['Client A', 'Client B', 'Client C']
}

# Convert the data to DataFrames for easier manipulation
invoices_df = pd.DataFrame(invoice_data)
bank_df = pd.DataFrame(bank_data)

# Convert DataFrames to dictionaries
invoices_dict = invoices_df.to_dict(orient='records')  # List of dictionaries
bank_dict = bank_df.to_dict(orient='records')  # List of dictionaries

# Agent for Invoice Scraping (simulating scraping from a website)
invoice_scraper_agent = Agent(
    role="Invoice Scraper",
    goal="Scrape invoices from a website and categorize them",
    backstory="This agent scrapes invoice data from a website and categorizes them into a structured format for further processing.",
    tools=[ScrapeWebsiteTool()],
    verbose=True
)

# Agent for Payment Matching (matches bank payments with invoices)
payment_matching_agent = Agent(
    role="Payment Matcher",
    goal="Match payments from bank with invoices",
    backstory="This agent matches payments recorded in the bank transactions to outstanding invoices, ensuring accurate financial records.",
    verbose=True
)

# Task to scrape invoices
scrape_invoices_task = Task(
    description="Scrape and extract invoices data from the website.",
    expected_output="Extracted invoice data.",
    agent=invoice_scraper_agent
)

# Task to match payments with invoices
match_payments_task = Task(
    description="Match payments made in the bank statement with invoices.",
    expected_output="Payment matched to the correct invoice.",
    agent=payment_matching_agent
)

# Creating the Crew (multi-agent collaboration)
financial_automation_crew = Crew(
    agents=[invoice_scraper_agent, payment_matching_agent],
    tasks=[scrape_invoices_task, match_payments_task],
    verbose=True
)

# Simulate scraping invoices (In practice, you would scrape a website for data)
result = financial_automation_crew.kickoff(inputs={'invoices': invoices_dict, 'bank_data': bank_dict})

# Displaying the results
matched_payments = pd.merge(invoices_df, bank_df, on='Client', how='inner')
print("Matched Payments:")
print(matched_payments)

# Automating Credit Control (send reminders for unpaid invoices)
unpaid_invoices = invoices_df[~invoices_df['Invoice Number'].isin(matched_payments['Invoice Number'])]
if not unpaid_invoices.empty:
    for index, row in unpaid_invoices.iterrows():
        print(f"Reminder: {row['Client']} - Invoice {row['Invoice Number']} is overdue by {row['Due Date']}.")
